# Review judge prompts and compare supplied summaries

Inspect the evaluation instructions, review a source and its summaries, and compare results while keeping the evaluator fixed.

**Default: offline.** Run every cell to inspect the judge prompts, check the bundled summary's structure, and save an experiment draft. No credentials are loaded and no model calls are made. Faithfulness and coverage scores are available after a deliberate live evaluation or loading genuine saved results.

The bundled material is fictional. The default baseline and candidate are the same prewritten summary, so they form a control. Supply your own summary variants for a meaningful comparison.

## 1. Choose inputs and keep live work off

Use the project's Python environment with the `notebooks` extra installed. Start Jupyter from this repository or its `notebooks` folder. Change the paths below to review your own approved inputs.

For a real prompt test, provide baseline and candidate summaries produced from the **same transcript and supplied context**. The default uses the same prewritten fixture for both: a control that cannot establish a prompt improvement.

In [ ]:
import hashlib
import html
import inspect
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, Markdown, display
from meeting_eval import BatchResult, MeetingSummaryEvaluator
from meeting_eval.comparison import compare_batches
from meeting_eval.diagnostics import check_structure, diagnostic_prompt
from meeting_eval.judge import JUDGE_SYSTEM_INSTRUCTION
from meeting_eval.metrics import COVERAGE_STEPS, CLARITY_STEPS, MeetingFaithfulnessTemplate
from meeting_eval.policy import METHODOLOGY_VERSION
from meeting_eval.provenance import evaluator_fingerprint
from meeting_eval.reports import write_csv

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file() and (path / "src/meeting_eval").is_dir())
RUN_LIVE = False
MODEL = None           # Live only: None uses OPENAI_MODEL or the package's documented default.
MAX_CONCURRENCY = 2
MEETING_ID = "meeting_001"
TRANSCRIPT_PATH = ROOT / "examples/data/transcripts/meeting_001.txt"
CONTEXT_PATH = ROOT / "examples/data/contexts/meeting_001.md"  # Or None if the producer received no context.
BASELINE_SUMMARY_PATH = ROOT / "examples/data/summaries/meeting_001.md"
CANDIDATE_SUMMARY_PATH = BASELINE_SUMMARY_PATH  # Change to your actual candidate summary file.
MANUAL_QA_PRESENT = True # This fixture contains substantive Q&A; review this label for other inputs.
SAVED_BASELINE_JSON = None  # Optional Path to a genuine baseline results.json.
SAVED_CANDIDATE_JSON = None # Optional Path to its compatible candidate results.json.

if os.environ.get("MEETING_EVAL_OFFLINE") == "1" and RUN_LIVE:
    raise RuntimeError("The learning runner requires offline mode. Leave RUN_LIVE=False.")
FROZEN_EVALUATOR_FINGERPRINT = evaluator_fingerprint()
print("Mode:", "LIVE opt-in" if RUN_LIVE else "OFFLINE — no credentials or model requests")
print("Evaluation methodology:", METHODOLOGY_VERSION)
print("Evaluator fingerprint:", FROZEN_EVALUATOR_FINGERPRINT)

## 2. Find the evaluation instructions

The **judge prompts** tell the evaluator how to assess a supplied summary. Keep the evaluator policy and code fixed when comparing summary variants so the comparison answers one question.

If independent human review later reveals a systematic judge error, treat that as a separate judge-calibration change and establish a new evaluation baseline.

In [ ]:
roles = [
    ("src/meeting_eval/policy.py", "Evaluation policy", "Versions, source authority, attribution, and labeled derived outreach. Keep fixed."),
    ("src/meeting_eval/metrics.py", "Metric judges", "Four faithfulness prompts, coverage steps/rubric, and optional clarity. Keep fixed."),
    ("src/meeting_eval/diagnostics.py", "Diagnostic judge and evidence", "Findings, source quotations, and deterministic structure checks. Keep fixed."),
    ("src/meeting_eval/judge.py", "Judge system message and adapter", "Untrusted-data boundary, strict structured responses, and request usage. Keep fixed."),
    ("src/meeting_eval/provenance.py", "Experiment identity", "Safe resolved settings and a fingerprint covering evaluator code, prompts, and schemas."),
]
display(Markdown("| File | Role | What to review |\n|---|---|---|\n" + "\n".join(
    f"| `{path}` | {role} | {purpose} |" for path, role, purpose in roles)))
print("Every listed source exists:", all((ROOT / path).is_file() for path, _, _ in roles))

## 3. Read a source, its separate context, and a prewritten summary

Notice the authority boundary: the transcript records what was said in the meeting. The supplied workshop context can clarify facts and identities, but cannot make a recommendation or discussion occur in that meeting.

In [ ]:
transcript = TRANSCRIPT_PATH.read_text(encoding="utf-8-sig")
provided_context = CONTEXT_PATH.read_text(encoding="utf-8-sig") if CONTEXT_PATH is not None else None
baseline_summary = BASELINE_SUMMARY_PATH.read_text(encoding="utf-8-sig")
candidate_summary = CANDIDATE_SUMMARY_PATH.read_text(encoding="utf-8-sig")

for label, text in (("Transcript", transcript), ("Explicitly provided context", provided_context),
                    ("Prewritten baseline summary", baseline_summary)):
    display(Markdown(f"### {label}"))
    if text is None:
        print("No context was supplied.")
    elif label.endswith("summary"):
        display(Markdown(text))
    else:
        print(text)
print("Prewritten summary inputs are identical:", baseline_summary == candidate_summary)
print("Record how your supplied variants were created in the experiment notes below.")

### Inspect one exact judge instruction

Choose another key below to inspect the claim-extraction source, verdict source, coverage/clarity steps, or system instruction. Opening these views makes no requests. The diagnostic view includes this loaded transcript, context, summary, and the evaluation policy.

In [ ]:
REVIEW_COMPONENT = "diagnostics_for_loaded_pair"
review_material = {
    "diagnostics_for_loaded_pair": diagnostic_prompt(transcript, baseline_summary, provided_context),
    "faithfulness_claim_extraction_source": inspect.getsource(MeetingFaithfulnessTemplate.generate_claims),
    "faithfulness_verdict_source": inspect.getsource(MeetingFaithfulnessTemplate.generate_verdicts),
    "coverage_steps": "\n\n".join(COVERAGE_STEPS),
    "clarity_steps": "\n\n".join(CLARITY_STEPS),
    "judge_system": JUDGE_SYSTEM_INSTRUCTION,
}
selected_instruction = review_material[REVIEW_COMPONENT]
print("Available views:", ", ".join(review_material))
print("Selected:", REVIEW_COMPONENT, "| Characters:", len(selected_instruction))
display(HTML("<details><summary>Open exact instruction text</summary>"
             "<pre style='white-space:pre-wrap;max-height:32rem;overflow:auto'>"
             + html.escape(selected_instruction) + "</pre></details>"))

## 4. Record the comparison hypothesis

Describe the intended difference between your supplied summaries and the human review that would establish an improvement. The example is a hypothesis, not a demonstrated result. The default identical inputs cannot demonstrate an improvement.

In [ ]:
EXPERIMENT_NAME = "check-number-unit-and-horizon"
HYPOTHESIS = "The candidate summary may reduce numeric distortions without dropping useful qualifications."
SUCCESS_CRITERION = "Human review finds fewer material numeric errors and no new material omissions or attribution errors."
REVIEW_NOTES = "Not reviewed yet. Default inputs are identical prewritten summaries."
assert evaluator_fingerprint() == FROZEN_EVALUATOR_FINGERPRINT
print("The evaluator fingerprint stayed fixed.")

### Make predictions before asking a judge

Use the loaded transcript to predict the treatment of these **exercise sentences**. They are not model outputs or scored results. The point is to identify the errors that matter when comparing summary variants.

In [ ]:
display(Markdown("""| Exercise sentence | What should a reviewer check? |
|---|---|
| “Short-term yields declined 10% this year.” | Value, unit, maturity, and horizon against Alex's 01:20 remark. |
| “The institution confirmed a rate cut next month.” | A participant question versus Jordan's qualified 06:35 answer. |
| “Suggested outreach: Explore the advisor's near-term liquidity needs.” | A labeled generated question versus a claimed meeting endorsement. |
| “The workshop identifier was CM-TEST-001.” | Separate context can support the identifier; it cannot establish that it was spoken in the meeting. |
"""))
display(HTML("<details><summary>Show the boundary reminders</summary>"
             "<p>Basis points are not percent returns; the cited yield move was weekly. "
             "The source gives no rate-cut date forecast. A labeled exploratory question can be "
             "derived from a supported insight, while an asserted recommendation needs explicit "
             "meeting support. The identifier appears only in provided context.</p></details>"))

## 5. Run the structural checks offline

These are real deterministic checks of headings, item limits, provenance labels, tags, and related format requirements. They do not assess whether a financial claim is supported or whether important meaning was omitted. `MANUAL_QA_PRESENT` above is a human-supplied label for this exercise.

In [ ]:
structure_result, structure_findings = check_structure(baseline_summary, MANUAL_QA_PRESENT)
checks = structure_result.details["checks"]
display(Markdown("| Format check | Result |\n|---|---|\n" + "\n".join(
    f"| `{name}` | {'Not determined' if passed is None else 'Satisfied' if passed else 'Needs attention'} |"
    for name, passed in checks.items())))
print("Executive bullet count:", structure_result.details["executive_bullet_count"])
for finding in structure_findings:
    print(finding.description)
print("Faithfulness: not run. Coverage: not run. Structural checks do not establish summary quality.")

## 6. Save an inspectable experiment draft

The draft saves the input identities, fixed evaluator fingerprint, and your hypothesis. It records that the summaries are supplied inputs. Local result files can contain source quotations; keep them in an appropriate local location when using real meetings.

In [ ]:
def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest() if text is not None else None

experiment_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
experiment_dir = ROOT / "results/summary_experiments" / experiment_id
experiment_dir.mkdir(parents=True, exist_ok=False)
manifest = {
    "experiment_id": experiment_id, "name": EXPERIMENT_NAME, "status": "offline_draft_no_quality_scores",
    "hypothesis": HYPOTHESIS, "success_criterion": SUCCESS_CRITERION,
    "human_review": {"status": "pending", "notes": REVIEW_NOTES},
    "evaluator_fingerprint": FROZEN_EVALUATOR_FINGERPRINT,
    "methodology_version": METHODOLOGY_VERSION,
    "meeting_id": MEETING_ID,
    "transcript": {"path": str(TRANSCRIPT_PATH), "sha256": text_sha256(transcript)},
    "provided_context": {"path": str(CONTEXT_PATH) if CONTEXT_PATH is not None else None,
                         "sha256": text_sha256(provided_context)},
    "summary_inputs": {
        "baseline": {"path": str(BASELINE_SUMMARY_PATH), "sha256": text_sha256(baseline_summary), "origin": "supplied_input"},
        "candidate": {"path": str(CANDIDATE_SUMMARY_PATH), "sha256": text_sha256(candidate_summary), "origin": "supplied_input"},
    },
    "evaluation_runs": None,
    "limits": "Fixture scores do not establish production improvement. Record how the supplied variants were created in human-review notes.",
}
manifest_path = experiment_dir / "experiment_manifest.json"

def save_manifest():
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")

save_manifest()
print("Saved experiment draft:", experiment_dir)
print("Summary inputs differ:", baseline_summary != candidate_summary)
print("Human review:", manifest["human_review"]["status"])

## 7. Optional live evaluation with one frozen judge

Set `RUN_LIVE=True` when ready to send these source/summary pairs to the configured model endpoint. This evaluates **two pairs**, usually **12 judge invocations** in total: four faithfulness stages, one coverage call, and one diagnostic call per pair, before any SDK retries. Full transcripts are repeatedly sent, so input length matters to cost and rate limits.

The same evaluator handles both variants. Each run writes its own detailed JSON, CSV, and HTML. A failed baseline stops the exercise before spending on the candidate. Authentication, schema, and timeout problems are execution problems, not bad-summary scores.

With the default identical prewritten summaries, score variation measures judge variation on a control.

In [ ]:
live_baseline = live_candidate = None
if RUN_LIVE:
    if os.environ.get("MEETING_EVAL_OFFLINE") == "1":
        raise RuntimeError("The learning runner requires offline mode.")
    if evaluator_fingerprint() != FROZEN_EVALUATOR_FINGERPRINT:
        raise RuntimeError("The evaluator changed. Restart this experiment and establish a new baseline.")
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
    resolved_judge_model = MODEL or os.getenv("OPENAI_MODEL", "gpt-6-luna")
    inputs_dir = experiment_dir / "evaluation_inputs"
    transcript_dir = inputs_dir / "transcripts"
    baseline_dir, candidate_dir = inputs_dir / "baseline_summaries", inputs_dir / "candidate_summaries"
    context_dir = inputs_dir / "contexts" if provided_context is not None else None
    for folder in (transcript_dir, baseline_dir, candidate_dir, context_dir):
        if folder is not None:
            folder.mkdir(parents=True, exist_ok=True)
    (transcript_dir / f"{MEETING_ID}.txt").write_text(transcript, encoding="utf-8")
    (baseline_dir / f"{MEETING_ID}.md").write_text(baseline_summary, encoding="utf-8")
    (candidate_dir / f"{MEETING_ID}.md").write_text(candidate_summary, encoding="utf-8")
    if context_dir is not None:
        (context_dir / f"{MEETING_ID}.md").write_text(provided_context, encoding="utf-8")
    manifest["status"] = "evaluating"
    manifest["evaluation_runs"] = {}
    save_manifest()
    try:
        async with MeetingSummaryEvaluator(model=resolved_judge_model, max_concurrency=MAX_CONCURRENCY,
                                           metrics=("faithfulness", "coverage"), env_file=None) as evaluator:
            live_baseline = await evaluator.aevaluate_batch(transcript_dir, baseline_dir,
                experiment_dir / "judge_runs", context_dir=context_dir)
            manifest["evaluation_runs"]["baseline"] = str(Path(live_baseline.output_dir) / "results.json")
            save_manifest()
            if any(result.execution_status != "completed" for result in live_baseline.results):
                raise RuntimeError("Baseline evaluation did not complete; inspect its saved report before continuing.")
            live_candidate = await evaluator.aevaluate_batch(transcript_dir, candidate_dir,
                experiment_dir / "judge_runs", context_dir=context_dir)
            manifest["evaluation_runs"]["candidate"] = str(Path(live_candidate.output_dir) / "results.json")
            if any(result.execution_status != "completed" for result in live_candidate.results):
                raise RuntimeError("Candidate evaluation did not complete; inspect its saved report.")
        manifest["status"] = "evaluated_pending_human_review"
        print("Baseline report:", Path(live_baseline.output_dir) / "report.html")
        print("Candidate report:", Path(live_candidate.output_dir) / "report.html")
    except Exception as exc:
        manifest["status"] = "evaluation_failed"
        manifest["error_type"] = type(exc).__name__
        raise RuntimeError("Evaluation stopped. Inspect the saved run reports and manifest before retrying.") from None
    finally:
        save_manifest()
else:
    print("OFFLINE/default: judge evaluation skipped. No faithfulness or coverage scores were created.")

## 8. Compare genuine saved runs

Live results above are used when available. Otherwise set `SAVED_BASELINE_JSON` and `SAVED_CANDIDATE_JSON` to two existing result files and rerun from the top. Loading them makes no model requests.

The package refuses comparisons with different evaluator policies, judge settings, versions, meeting IDs, transcripts, or supplied context. Never relax those checks just to obtain a delta. Incomplete evaluations remain visible and do not enter aggregate deltas.

A saved pair may come from another experiment. Its recorded sources and judge settings control that comparison; loading it does not establish a relationship to the summary inputs selected above.

In [ ]:
comparison = None
baseline_batch = live_baseline
candidate_batch = live_candidate
if baseline_batch is None and candidate_batch is None and (SAVED_BASELINE_JSON is not None or SAVED_CANDIDATE_JSON is not None):
    if SAVED_BASELINE_JSON is None or SAVED_CANDIDATE_JSON is None:
        raise ValueError("Select both baseline and candidate results.json paths.")
    saved_paths = [Path(SAVED_BASELINE_JSON), Path(SAVED_CANDIDATE_JSON)]
    if all(path.is_file() for path in saved_paths):
        baseline_batch, candidate_batch = [BatchResult.model_validate_json(path.read_text(encoding="utf-8")) for path in saved_paths]
    else:
        print("A selected results.json does not exist. Correct the paths; no comparison was fabricated.")
if baseline_batch is not None and candidate_batch is not None:
    comparison = compare_batches(baseline_batch, candidate_batch)
    comparison_path = experiment_dir / "comparison.json"
    comparison_path.write_text(json.dumps(comparison, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    write_csv(comparison["rows"], experiment_dir / "comparison.csv")
    manifest["comparison"] = {"path": str(comparison_path), "source": "live_experiment" if RUN_LIVE else "selected_saved_runs",
                              "baseline_run_id": baseline_batch.run_id, "candidate_run_id": candidate_batch.run_id,
                              "relationship_to_current_inputs": "Evaluated the selected summary inputs" if RUN_LIVE else "Not established for externally saved runs"}
    save_manifest()
    print("Complete paired evaluations:", comparison["complete_pair_count"], "/", comparison["meeting_count"])
    for name, values in comparison["metric_deltas"].items():
        print(name, "— paired count:", values["paired_count"], "mean delta:", values["mean_delta"], "median delta:", values["median_delta"])
    print("Comparison saved:", comparison_path)
    display(Markdown("Positive deltas favor the candidate **for these recorded judgments**. Review material findings, not just score movement."))
else:
    print("No genuine saved result pair selected, so no quality scores or deltas are displayed.")
    print("After an evaluation, select its two results.json files above. Draft location:", experiment_dir)

## 9. Decide what changed, then test it elsewhere

Before looking at scores, have a human read the same sources and both summaries without the variant labels. Record the material issues, useful paraphrases, omissions, and uncertainty independently. Then reconcile the human review with the diagnostic findings.

For each paired case, record:

| Review question | Your note |
|---|---|
| Which material issue was actually fixed? | Cite the source and changed summary wording. |
| Which new material issue appeared? | Include regressions even when the mean score rose. |
| Which finding was a false alarm or a missed error? | Keep human evidence separate from judge suggestions. |
| How were the variants created? | Record the source of each summary and distinguish generated, edited, and identical controls. |
| Is the result repeatable? | Repeat noisy cases under the same judge and inspect the variation. |
| Does the change generalize? | Test a held-out set that did not drive the edit; include long, noisy, and subtle financial cases. |

Do not tune to a single fictional meeting. Use the calibration guidance in `docs/calibration.md` for independent labels and representative cases. If you change judge rules, model settings, or methodology, establish a new baseline rather than treating it as the same summary comparison. A one-pair local test cannot establish production-wide improvement.

In [ ]:
print("Notebook completed in", "LIVE opt-in mode" if RUN_LIVE else "OFFLINE mode")
print("Experiment manifest:", manifest_path)
print("Human review remains:", manifest["human_review"]["status"])
if not RUN_LIVE:
    print("Credentials were not loaded by this notebook; judge calls were skipped.")
    print("Observed offline work: source inspection, judge prompts, input hashes, and deterministic format checks.")
if comparison is None:
    print("No claim of score improvement has been made.")